# Three ROIs plot of SWEEP2 data + Idle/Running state
------------------------------------
The script create three different ROIs located in anterior, mid and back part of the brain then it compute different metrics on the three sposts: 
1) Compare MD values across scanner states 
2) Compute tSNR on b0s between the regions 
3) Compare the averagae signal in b1000, b1500, b2800 between sessions 

# Create MASKs
--------------

In [ ]:
import os 
import numpy as np 
import subprocess as sub 
import nibabel as nib
import matplotlib.pyplot as plt
import nilearn as nil 
from nilearn.image.image import mean_img #Estimate mean of a 4D volume --> mean_haxby = mean_img(func_filename)
from nilearn.plotting import plot_roi, show
from nilearn.maskers import NiftiMasker #Create a mask


vps=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43] 
sessions=["ses-01","ses-02"] #,"ses-03","ses-04","ses-05","ses-06"

APs = ["A", "B", "C"]

Scanner_States = r"/home/malberti/Unix_Folders/SWEEP2/Script/Statistical_Analysis/Scanner_State.ods" # The ods file contain all the info related to session, scanner_State etc... It is not used at the moment

# To make it easiers:  --> The script create a mask in dMRI space, and MNI space 
NN_INTERLEAVED  =  ["sub-00", "sub-07","sub-18", "sub-21" ,"sub-22" ,"sub-29", "sub-34", "sub-35", "sub-40", "sub-41"] # Non interleaved
FIRST_PARTICIPANTs = ["sub-01", "sub-03","sub-05", "sub-08" ,"sub-10" ,"sub-12", "sub-14", "sub-16", "sub-19", "sub-23", "sub-25", "sub-27", "sub-30", "sub-36", "sub-38", "sub-42"]    # "sub-32", is missing because of artifacats
SECOND_PARTICIPANTs = ["sub-02", "sub-04","sub-06", "sub-09" ,"sub-11" ,"sub-13", "sub-15", "sub-17", "sub-20", "sub-24", "sub-26", "sub-28", "sub-31", "sub-33", "sub-37", "sub-39", "sub-43"]

hemis=["lh" ,"rh"]
cube_imgs = {}   # collect this participant's cubes here, keyed by hemi, so we can plot both hemis together after the loop
# The script compute and plot average MD signal in A, B and C differentiating between scanner state and APs, each box plot, represent the difference between scanner states : 
# --> ses-01: A ws ses-02:A --> anterior, mid and posterior 
# --> ses-01: B ws ses-02:B --> anterior, mid and posterior
# --> ses-01: C ws ses-02:C --> anterior, mid and posterior

In [ ]:
#FIrst create participants maska and plot them
for vp in vps: 
     for session in sessions: 
        for AP in APs: 
            id = f"sub-{vp:02d}"   # id: standard subject ID (sub-XX)
            subjid=f"{id}_{session}"
        
            derivatives = r"/home/malberti/Unix_Folders/SWEEP2/derivatives"
            mask_IMG=nib.load(os.path.join(derivatives, id, session, "dwi", "eddy", f"topup_out_{AP}_unwarped_{AP}_brain_mask.nii"))
            ref_b0s_IMG=nib.load(os.path.join(derivatives, id, session, "dwi", "eddy", f"topup_out_{AP}_unwarped_{AP}_brain.nii"))
            
            mask=mask_IMG.get_fdata()
            ref_b0s=ref_b0s_IMG.get_fdata()
            ref_b0=mean_img(ref_b0s_IMG)
            affine = mask_IMG.affine  # keep original affine
            
            """
            X//2 - px//2 --> move the ROI on left/right axis
            Y//4  --> move the ROI anterior to post
            Z//2 --> Up/down
            """

            # ----------------------------
            # Parameters
            # ----------------------------
            X, Y, Z = mask.shape[:3]          # total volume size
            cube_size = (15, 10, 10)          # 10x10x10 cube
            px, py, pz = cube_size
            
            
            # ===========================================================================
            ## --> FRONT ROI 
            # Start cube at center along Z (axial) and roughly middle in X/Y
            cube_start_front = (X//2-10, Y//4 + 50, Z//2)  # axial posterior half

            # ----------------------------
            # Create new mask with single cube
            # ----------------------------
            cube_mask_front = np.zeros((X, Y, Z), dtype=np.uint8)
            cube_mask_front[
                cube_start_front[0]:cube_start_front[0]+px,
                cube_start_front[1]:cube_start_front[1]+py,
                cube_start_front[2]:cube_start_front[2]+pz
            ] = 1

            # Optionally, multiply by original mask to stay inside brain
            cube_mask_front = cube_mask_front * mask

            cube_mask_front_img = nib.Nifti1Image(cube_mask_front, affine)
            
            # ===========================================================================
            ## --> BACK ROI 
            # Start cube at center along Z (axial) and roughly middle in X/Y
            cube_start_back = (X//2-10, Y//4 - 3, Z//2)  # axial posterior half
            
            cube_mask_back = np.zeros((X, Y, Z), dtype=np.uint8)
            cube_mask_back[
                cube_start_back[0]:cube_start_back[0]+px,
                cube_start_back[1]:cube_start_back[1]+py,
                cube_start_back[2]:cube_start_back[2]+pz
            ] = 1

            # Optionally, multiply by original mask to stay inside brain
            cube_mask_back = cube_mask_back * mask

            # ----------------------------
            # Save new mask
            # ----------------------------
            cube_mask_back_img = nib.Nifti1Image(cube_mask_back, affine)
            
            # ===========================================================================
            ## --> MID ROI 
            # Start cube at center along Z (axial) and roughly middle in X/Y
            cube_start_mid = (X//2-10, Y//4 + 25 , Z//2 + 12)  # axial posterior half
            
            cube_mask_mid = np.zeros((X, Y, Z), dtype=np.uint8)
            cube_mask_mid[
                cube_start_mid[0]:cube_start_mid[0]+px,
                cube_start_mid[1]:cube_start_mid[1]+py,
                cube_start_mid[2]:cube_start_mid[2]+pz
            ] = 1

            # Optionally, multiply by original mask to stay inside brain
            cube_mask_mid = cube_mask_mid * mask

            # ----------------------------
            # Save new mask
            # ----------------------------
            cube_mask_mid_img = nib.Nifti1Image(cube_mask_mid, affine)
            
            # ===========================================================================
            # SAVE & PLOT
            # ===========================================================================

            display = plot_roi(
                ref_b0,
                bg_img=None,
                display_mode='ortho',
                draw_cross=False,
                threshold=0,
                black_bg=True,
                cmap="Grays_r",
                colorbar=False
            )

            print(f"\nMask: {subjid} || Set: {AP}")
            display.add_overlay(cube_mask_front_img, cmap='BrBG')
            display.add_overlay(cube_mask_back_img, cmap='Blues_r') #cube_mask_back_img
            display.add_overlay(cube_mask_mid_img, cmap='RdGy') #cube_mask_back_img

            show()

            masks_output = os.path.join(derivatives, id, session, "dwi", "masks")
            os.makedirs(masks_output, exist_ok=True)
            
            nib.save(cube_mask_front_img, (os.path.join(masks_output, f"{subjid}_{AP}_cube_mask-front.nii.gz")))
            nib.save(cube_mask_mid_img, (os.path.join(masks_output, f"{subjid}_{AP}_cube_mask-mid.nii.gz")))
            nib.save(cube_mask_back_img, (os.path.join(masks_output, f"{subjid}_{AP}_cube_mask-back.nii.gz")))


### Make ROIs in each hemisphere 
----------------------------

In [ ]:
#FIrst create participants maska and plot them
for vp in vps: 
     for session in sessions: 
        for AP in APs: 
            id = f"sub-{vp:02d}"   # id: standard subject ID (sub-XX)
            subjid=f"{id}_{session}"
        
            derivatives = r"/home/malberti/Unix_Folders/SWEEP2/derivatives"
            mask_IMG=nib.load(os.path.join(derivatives, id, session, "dwi", "eddy", f"topup_out_{AP}_unwarped_{AP}_brain_mask.nii"))
            ref_b0s_IMG=nib.load(os.path.join(derivatives, id, session, "dwi", "eddy", f"topup_out_{AP}_unwarped_{AP}_brain.nii"))
            
            mask=mask_IMG.get_fdata()
            ref_b0s=ref_b0s_IMG.get_fdata()
            ref_b0=mean_img(ref_b0s_IMG)
            affine = mask_IMG.affine  # keep original affine
            
            """
            X//2 - px//2 --> move the ROI on left/right axis
            Y//4  --> move the ROI anterior to post
            Z//2 --> Up/down
            """
            hemis=["lh" ,"rh"]

            for hemi in hemis:
                # ----------------------------
                # Parameters
                # ----------------------------
                X, Y, Z = mask.shape[:3]          # total volume size
                cube_size = (10, 10, 10)          # 15x15x15 cube
                px, py, pz = cube_size

                # cube coordinates, so lh/rh produced the exact same cube twice.
                # hemi_shift moves the cube away from the midline (X//2) to the
                # left for "lh" and to the right for "rh".
                hemi_shift = 11 if hemi == "lh" else -16

                # ===========================================================================
                ## --> FRONT ROI 
                # Start cube at center along Z (axial) and roughly middle in X/Y
                cube_start_front = (X//2+hemi_shift, Y//4 + 47, Z//2)  # axial posterior half

                # ----------------------------
                # Create new mask with single cube
                # ----------------------------
                cube_mask_front = np.zeros((X, Y, Z), dtype=np.uint8)
                cube_mask_front[
                    cube_start_front[0]:cube_start_front[0]+px,
                    cube_start_front[1]:cube_start_front[1]+py,
                    cube_start_front[2]:cube_start_front[2]+pz
                ] = 1

                # Optionally, multiply by original mask to stay inside brain
                cube_mask_front = cube_mask_front * mask

                cube_mask_front_img = nib.Nifti1Image(cube_mask_front, affine)
                
                # ===========================================================================
                ## --> BACK ROI 
                # Start cube at center along Z (axial) and roughly middle in X/Y
                cube_start_back = (X//2+hemi_shift-2, Y//4-1, Z//2-3)  # axial posterior half
                
                cube_mask_back = np.zeros((X, Y, Z), dtype=np.uint8)
                cube_mask_back[
                    cube_start_back[0]:cube_start_back[0]+px,
                    cube_start_back[1]:cube_start_back[1]+py,
                    cube_start_back[2]:cube_start_back[2]+pz
                ] = 1

                # Optionally, multiply by original mask to stay inside brain
                cube_mask_back = cube_mask_back * mask

                # ----------------------------
                # Save new mask
                # ----------------------------
                cube_mask_back_img = nib.Nifti1Image(cube_mask_back, affine)
                
                # ===========================================================================
                ## --> MID ROI 
                # Start cube at center along Z (axial) and roughly middle in X/Y
                cube_start_mid = (X//2+hemi_shift, Y//4 + 20 , Z//2 + 9)  # axial posterior half
                
                cube_mask_mid = np.zeros((X, Y, Z), dtype=np.uint8)
                cube_mask_mid[
                    cube_start_mid[0]:cube_start_mid[0]+px,
                    cube_start_mid[1]:cube_start_mid[1]+py,
                    cube_start_mid[2]:cube_start_mid[2]+pz
                ] = 1

                # Optionally, multiply by original mask to stay inside brain
                cube_mask_mid = cube_mask_mid * mask

                # ----------------------------
                # Save new mask
                # ----------------------------
                cube_mask_mid_img = nib.Nifti1Image(cube_mask_mid, affine)
                
                # store this hemi's cubes so lh and rh can be plotted together, below, after the hemi loop
                cube_imgs[hemi] = {
                    "front": cube_mask_front_img,
                    "mid": cube_mask_mid_img,
                    "back": cube_mask_back_img,
                }
 

            # ===========================================================================
            # SAVE & PLOT — both hemispheres together
            # ===========================================================================
 
            display = plot_roi(
                ref_b0,
                bg_img=None,
                display_mode='mosaic',
                draw_cross=False,
                threshold=0,
                black_bg=True,
                cmap="Grays_r",
                colorbar=False,
                title=f"Mask: {subjid} || Set: {AP}"
            )
 
            print(f"\nMask: {subjid} || Set: {AP} || Hemis: lh + rh")
            for hemi in hemis:
                display.add_overlay(cube_imgs[hemi]["front"], cmap='BrBG')
                display.add_overlay(cube_imgs[hemi]["back"], cmap='Blues_r') #cube_mask_back_img
                display.add_overlay(cube_imgs[hemi]["mid"], cmap='RdGy') #cube_mask_back_img
 
            show()
 
            masks_output = os.path.join(derivatives, id, session, "dwi", "masks")
            os.makedirs(masks_output, exist_ok=True)
            for hemi in hemis:
                pass
                nib.save(cube_imgs[hemi]["front"], (os.path.join(masks_output, f"{subjid}_{AP}_cube_mask-front_{hemi}.nii.gz")))
                nib.save(cube_imgs[hemi]["mid"], (os.path.join(masks_output, f"{subjid}_{AP}_cube_mask-mid_{hemi}.nii.gz")))
                nib.save(cube_imgs[hemi]["back"], (os.path.join(masks_output, f"{subjid}_{AP}_cube_mask-back_{hemi}.nii.gz")))
 


### Create ROIs in MNI space
----------------------------------

In [ ]:
derivatives = r"/home/malberti/Unix_Folders/SWEEP2/derivatives"
mask_IMG=nib.load(r"/home/malberti/Unix_Folders/SWEEP2/Script/DEWEY_v6/Atlases/MNI_icbm_152_Template/tpl-MNI152NLin2009cAsym_res-02_desc-brain_mask.nii.gz")
ref_b0s_IMG=nib.load(r"/home/malberti/Unix_Folders/SWEEP2/Script/DEWEY_v6/Atlases/MNI_icbm_152_Template/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz")

mask=mask_IMG.get_fdata()
ref_b0s=ref_b0s_IMG.get_fdata()
ref_b0=mean_img(ref_b0s_IMG)
affine = mask_IMG.affine  # keep original affine

"""
X//2 - px//2 --> move the ROI on left/right axis
Y//4  --> move the ROI anterior to post
Z//2 --> Up/down
"""

# ----------------------------
# Parameters
# ----------------------------
X, Y, Z = mask.shape[:3]          # total volume size
cube_size = (15, 10, 10)          # 10x10x10 cube
px, py, pz = cube_size


# ===========================================================================
## --> FRONT ROI 
# Start cube at center along Z (axial) and roughly middle in X/Y
cube_start_front = (X//2-10, Y//4 + 50, Z//2)  # axial posterior half

# ----------------------------
# Create new mask with single cube
# ----------------------------
cube_mask_front = np.zeros((X, Y, Z), dtype=np.uint8)
cube_mask_front[
    cube_start_front[0]:cube_start_front[0]+px,
    cube_start_front[1]:cube_start_front[1]+py,
    cube_start_front[2]:cube_start_front[2]+pz
] = 1

# Optionally, multiply by original mask to stay inside brain
cube_mask_front = cube_mask_front * mask

cube_mask_front_img = nib.Nifti1Image(cube_mask_front, affine)

# ===========================================================================
## --> BACK ROI 
# Start cube at center along Z (axial) and roughly middle in X/Y
cube_start_back = (X//2-10, Y//4 - 3, Z//2)  # axial posterior half

cube_mask_back = np.zeros((X, Y, Z), dtype=np.uint8)
cube_mask_back[
    cube_start_back[0]:cube_start_back[0]+px,
    cube_start_back[1]:cube_start_back[1]+py,
    cube_start_back[2]:cube_start_back[2]+pz
] = 1

# Optionally, multiply by original mask to stay inside brain
cube_mask_back = cube_mask_back * mask

# ----------------------------
# Save new mask
# ----------------------------
cube_mask_back_img = nib.Nifti1Image(cube_mask_back, affine)

# ===========================================================================
## --> MID ROI 
# Start cube at center along Z (axial) and roughly middle in X/Y
cube_start_mid = (X//2-10, Y//4 + 25 , Z//2 + 12)  # axial posterior half

cube_mask_mid = np.zeros((X, Y, Z), dtype=np.uint8)
cube_mask_mid[
    cube_start_mid[0]:cube_start_mid[0]+px,
    cube_start_mid[1]:cube_start_mid[1]+py,
    cube_start_mid[2]:cube_start_mid[2]+pz
] = 1

# Optionally, multiply by original mask to stay inside brain
cube_mask_mid = cube_mask_mid * mask

# ----------------------------
# Save new mask
# ----------------------------
cube_mask_mid_img = nib.Nifti1Image(cube_mask_mid, affine)

# ===========================================================================
# SAVE & PLOT
# ===========================================================================

display = plot_roi(
    ref_b0,
    bg_img=None,
    display_mode='ortho',
    draw_cross=False,
    threshold=0,
    black_bg=True,
    cmap="Grays_r",
    colorbar=False
)

print(f"\nMask: {subjid} || Set: {AP}")
display.add_overlay(cube_mask_front_img, cmap='BrBG')
display.add_overlay(cube_mask_back_img, cmap='Blues_r') #cube_mask_back_img
display.add_overlay(cube_mask_mid_img, cmap='RdGy') #cube_mask_back_img

show()

masks_output = r"/home/malberti/Unix_Folders/SWEEP2/Script/Miscellaneous"
os.makedirs(masks_output, exist_ok=True)

nib.save(cube_mask_front_img, (os.path.join(masks_output, f"HCPex_MNI_cube_mask-front.nii.gz")))
nib.save(cube_mask_mid_img, (os.path.join(masks_output, f"HCPex_MNI_cube_mask-mid.nii.gz")))
nib.save(cube_mask_back_img, (os.path.join(masks_output, f"HCPex_MNI_cube_mask-back.nii.gz")))


# PLOT changes across Anterior - Mid and Posteriror brain 

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Config (as provided) ────────────────────────────────────────────────────
derivatives = r"/home/malberti/Unix_Folders/SWEEP2/derivatives"

vps = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20,
       21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 33, 34, 35, 36, 37, 38, 39,
       40, 41, 42, 43]
sessions = ["ses-01", "ses-02"]
APs = ["A", "B", "C"]
metric = "MD"

NN_INTERLEAVED      = ["sub-00", "sub-07", "sub-18", "sub-21", "sub-22", "sub-29", "sub-34", "sub-35", "sub-40", "sub-41"]
FIRST_PARTICIPANTs  = ["sub-01", "sub-03", "sub-05", "sub-08", "sub-10", "sub-12", "sub-14", "sub-16", "sub-19", "sub-23", "sub-25", "sub-27", "sub-30", "sub-36", "sub-38", "sub-42"]
SECOND_PARTICIPANTs = ["sub-02", "sub-04", "sub-06", "sub-09", "sub-11", "sub-13", "sub-15", "sub-17", "sub-20", "sub-24", "sub-26", "sub-28", "sub-31", "sub-33", "sub-37", "sub-39", "sub-43"]

condition_groups = {
    "Non-interleaved":     NN_INTERLEAVED,
    "First participants":  FIRST_PARTICIPANTs,
    "Second participants": SECOND_PARTICIPANTs,
}
condition_order = list(condition_groups.keys())
roi_names = ["front", "mid", "back"]

# ── Collect drift values ─────────────────────────────────────────────────────
records = []
hemi = "rh"
for vp in vps:
    id = f"sub-{vp:02d}"

    condition = None
    for label, ids in condition_groups.items():
        if id in ids:
            condition = label
            break  

    for AP in APs:
        roi_means = {ses: {} for ses in sessions}
        ok = True

        for session in sessions:
            # NOTE: assumed metric-map naming, mirroring the per-AP mask naming.
            # Adjust this path if your actual layout differs.
            m_path = os.path.join(derivatives, id, session, "dwi", "index", "Tensor", f"{id}_{session}_eddy-current_signal-drift_{AP}_corr_{metric}.nii")
            if not os.path.exists(m_path):
                print(f"✗  Missing metric map: {m_path}")
                ok = False
                break
            metric_data = nib.load(m_path).get_fdata()

            for roi in roi_names:
                mk_path = os.path.join(derivatives, id, session, "dwi", "masks", f"{id}_{session}_{AP}_cube_mask-{roi}_{hemi}.nii.gz")
                if not os.path.exists(mk_path):
                    print(f"✗  Missing mask: {mk_path}")
                    ok = False
                    continue
                mask = nib.load(mk_path).get_fdata().astype(bool)
                roi_means[session][roi] = np.mean(metric_data[mask])

        if not ok:
            continue

        for roi in roi_names:
            if roi in roi_means["ses-01"] and roi in roi_means["ses-02"]:
                drift = roi_means["ses-02"][roi] - roi_means["ses-01"][roi]
                records.append({"subject": id, "AP": AP, "condition": condition,
                                 "ROI": roi, "drift": drift})

df = pd.DataFrame(records)
print(f"\nCollected {len(df)} rows | subjects: {df['subject'].nunique()}")

# ── Plot: one box plot per AP ─────────────────────────────────────────────────
for AP in APs:
    sub_df = df[df["AP"] == AP]
    if sub_df.empty:
        print(f"No data for AP {AP}, skipping plot")
        continue

    fig, ax = plt.subplots(figsize=(9, 6))
    sns.boxplot(data=sub_df, x="condition", y="drift", hue="ROI",
                order=condition_order, hue_order=roi_names, ax=ax)
    sns.stripplot(data=sub_df, x="condition", y="drift", hue="ROI",
                order=condition_order, hue_order=roi_names,
                dodge=True, palette={roi: "black" for roi in roi_names},
                alpha=0.5, size=4,
                ax=ax, legend=False)

    ax.axhline(0, color="grey", linestyle="--", linewidth=1)
    ax.set_xlabel("")
    ax.set_ylim(0.0001, -0.0001)
    ax.set_ylabel(f"{metric} drift (ses-02 − ses-01)")
    ax.set_title(f"{metric} drift by condition and ROI | hemi = {hemi}  |  AP = {AP}")
    ax.legend(title="ROI", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


    # ── Plot: one box plot per AP ─────────────────────────────────────────────────
roi_plot_order = ["mid", "front", "back"]   # order requested: mid, ant/front, back

for AP in APs:
    sub_df = df[df["AP"] == AP]
    if sub_df.empty:
        print(f"No data for AP {AP}, skipping plot")
        continue

    fig, ax = plt.subplots(figsize=(9, 6))
    sns.boxplot(data=sub_df, x="ROI", y="drift", hue="condition",
                order=roi_plot_order, hue_order=condition_order, ax=ax)
    sns.stripplot(data=sub_df, x="ROI", y="drift", hue="condition",
                  order=roi_plot_order, hue_order=condition_order,
                  dodge=True, palette={c: "black" for c in condition_order},
                  alpha=0.5, size=4,
                  ax=ax, legend=False)

    ax.axhline(0, color="grey", linestyle="--", linewidth=1)
    ax.set_xlabel("")
    ax.set_ylabel(f"{metric} drift (ses-02 − ses-01)")
    ax.set_ylim(0.0001, -0.0001)
    ax.set_title(f"{metric} drift by ROI and condition | hemi = {hemi}  |  AP = {AP}")
    ax.legend(title="Condition", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

# Plot bval specific drift
----------------------------------

Instead of computing and plotting average on MD/FA etc... 
I want to compare different bval and differente signal distribution within ROIs 

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Config (as provided) ────────────────────────────────────────────────────
derivatives = r"/home/malberti/Unix_Folders/SWEEP2/derivatives"

vps = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43]

sessions = ["ses-01", "ses-02"]
APs = ["A", "B", "C"]
metric = "MD"
BVAL_TOL = 20  # b-values within this range are grouped into the same shell

NN_INTERLEAVED      = ["sub-00", "sub-07", "sub-18", "sub-21", "sub-22", "sub-29", "sub-34", "sub-35", "sub-40", "sub-41"]
FIRST_PARTICIPANTs  = ["sub-01", "sub-03", "sub-05", "sub-08", "sub-10", "sub-12", "sub-14", "sub-16", "sub-19", "sub-23", "sub-25", "sub-27", "sub-30", "sub-36", "sub-38", "sub-42"]
SECOND_PARTICIPANTs = ["sub-02", "sub-04", "sub-06", "sub-09", "sub-11", "sub-13", "sub-15", "sub-17", "sub-20", "sub-24", "sub-26", "sub-28", "sub-31", "sub-33", "sub-37", "sub-39", "sub-43"]



condition_groups = {
    "Non-interleaved":     NN_INTERLEAVED,
    "First participants":  FIRST_PARTICIPANTs,
    "Second participants": SECOND_PARTICIPANTs,
}

condition_order = list(condition_groups.keys())
roi_names = ["front", "mid", "back"]


def parse_bvals(bval_path):
    """Return 1-D array of b-values; handle space- or newline-separated files."""
    with open(bval_path) as f:
        content = f.read().split()
    return np.array([float(v) for v in content])


# ── Collect signal values (per subject, session, AP, bval-shell, ROI) ─────────
records = []

for vp in vps:
    id = f"sub-{vp:02d}"

    condition = None
    for label, ids in condition_groups.items():
        if id in ids:
            condition = label
            break
   
    for AP in APs:
        # roi_means[session][bval_shell][roi] = mean signal
        roi_means = {ses: {} for ses in sessions}

        for session in sessions:
            BVALs = os.path.join(derivatives, id, session, "dwi", "signal_drift", f"{id}_{session}_dwi_eddy_corrected_{AP}_noPA.bval")
            img_path = os.path.join(derivatives, id, session, "dwi", "signal_drift", f"{id}_{session}_dwi_eddy_corrected_{AP}_noPA.nii")

            bvals = parse_bvals(BVALs)
            metric_data = nib.load(img_path).get_fdata()   # shape (X, Y, Z, n_volumes)


            shells = np.unique(np.round(bvals / BVAL_TOL) * BVAL_TOL)
            roi_means[session] = {shell: {} for shell in shells}

            for roi in roi_names:
                mk_path = os.path.join(derivatives, id, session, "dwi", "masks", f"{id}_{session}_{AP}_cube_mask-{roi}.nii.gz")
               
                mask = nib.load(mk_path).get_fdata().astype(bool)

                for shell in shells:
                    vol_idx = np.abs(bvals - shell) <= BVAL_TOL
                    vols = metric_data[..., vol_idx]   # (X, Y, Z, n_vols_at_shell)
                    vals = vols[mask]                   # (n_voxels, n_vols_at_shell)
                    roi_means[session][shell][roi] = np.mean(vals)


        common_shells = set(roi_means["ses-01"].keys()) & set(roi_means["ses-02"].keys())

        for shell in sorted(common_shells):
            for roi in roi_names:
                if roi in roi_means["ses-01"][shell] and roi in roi_means["ses-02"][shell]:
                    val_ses1 = roi_means["ses-01"][shell][roi]
                    val_ses2 = roi_means["ses-02"][shell][roi]
                    records.append({"subject": id, "AP": AP, "condition": condition,
                                     "ROI": roi, "bval": shell,
                                     "signal_ses1": val_ses1, "signal_ses2": val_ses2,
                                     "drift": val_ses1 - val_ses2})

df = pd.DataFrame(records)
print(f"\nCollected {len(df)} rows | subjects: {df['subject'].nunique()}")

In [ ]:
# ── Plot: signal vs b-value, one panel per ROI (mid, front, back) ─────────────
roi_plot_order = ["mid", "front", "back"]

long_df = pd.melt(df, id_vars=["subject", "AP", "condition", "ROI", "bval"],
                   value_vars=["signal_ses1", "signal_ses2"],
                   var_name="session", value_name="signal")
long_df["session"] = long_df["session"].map({"signal_ses1": "ses-01", "signal_ses2": "ses-02"})
print(long_df)

In [ ]:

for AP in APs:
    fig, axes = plt.subplots(1, len(roi_plot_order), figsize=(15, 5), dpi=300, sharey=True)
    for ax, roi in zip(axes, roi_plot_order):
        sub_df = long_df[(long_df["ROI"] == roi) & (long_df["AP"] == AP)]
        sns.lineplot(data=sub_df, x="bval", y="signal", hue="condition",
                     style="session", hue_order=condition_order, alpha = 0.8,
                     marker="o", errorbar="se", ax=ax)
        ax.set_title(roi)
        ax.set_xlabel("b-value")
        if roi != roi_plot_order[0] and ax.get_legend():
            ax.legend_.remove()

    axes[0].set_ylabel(f"{metric} signal")
    fig.suptitle(f"{metric} signal vs b-value by ROI  |  AP = {AP}")
    plt.tight_layout()
    plt.show()



In [ ]:

# ── Plot: signal vs b-value, one panel per session, all ROIs + conditions together ──
for AP in APs:
    fig, axes = plt.subplots(1, len(sessions), figsize=(14, 6),dpi=300, sharey=True)

    for ax, session in zip(axes, sessions):
        sub_df = long_df[(long_df["AP"] == AP) & (long_df["session"] == session)]
        sns.lineplot(data=sub_df, x="bval", y="signal",
                     hue="condition", hue_order=condition_order,
                     style="ROI", style_order=roi_plot_order, alpha = 0.8,
                     marker="o", errorbar="se", ax=ax)
        ax.set_title(session)
        ax.set_xlabel("b-value")
        if session != sessions[0] and ax.get_legend():
            ax.legend_.remove()

    axes[0].set_ylabel(f"{metric} signal")
    axes[-1].legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.suptitle(f"{metric} signal vs b-value  |  AP = {AP}")
    plt.tight_layout()
    plt.show()


In [ ]:

# ── Plot: signal DIFFERENCE between sessions (decay curve), one panel per ROI ──
for AP in APs:
    fig, axes = plt.subplots(1, len(roi_plot_order), figsize=(15, 5), dpi=300, sharey=True)
    for ax, roi in zip(axes, roi_plot_order):
        sub_df = df[(df["ROI"] == roi) & (df["AP"] == AP)]
        sns.lineplot(data=sub_df, x="bval", y="drift", hue="condition",
                     hue_order=condition_order, alpha=0.8,
                     marker="o", errorbar="sd", ax=ax)
        ax.axhline(0, color="gray", linestyle="--", linewidth=1)
        ax.set_title(roi)
        ax.set_xlabel("b-value")
        if roi != roi_plot_order[0] and ax.get_legend():
            ax.legend_.remove()

    axes[0].set_ylabel(f"Signal drift (ses-02 − ses-01)")
    fig.suptitle(f"Session-to-session signal drift vs b-value  |  AP = {AP}")
    plt.tight_layout()
    plt.show()

# MNIs space plots
---------------------

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Config (as provided) ────────────────────────────────────────────────────
derivatives = r"/home/malberti/Unix_Folders/SWEEP2/derivatives"

vps = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20,
       21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 33, 34, 35, 36, 37, 38, 39,
       40, 41, 42, 43]
sessions = ["ses-01", "ses-02"]
APs = ["A", "B", "C"]
metric = "MD"

NN_INTERLEAVED      = ["sub-00", "sub-07", "sub-18", "sub-21", "sub-22", "sub-29", "sub-34", "sub-35", "sub-40", "sub-41"]
FIRST_PARTICIPANTs  = ["sub-01", "sub-03", "sub-05", "sub-08", "sub-10", "sub-12", "sub-14", "sub-16", "sub-19", "sub-23", "sub-25", "sub-27", "sub-30", "sub-36", "sub-38", "sub-42"]
SECOND_PARTICIPANTs = ["sub-02", "sub-04", "sub-06", "sub-09", "sub-11", "sub-13", "sub-15", "sub-17", "sub-20", "sub-24", "sub-26", "sub-28", "sub-31", "sub-33", "sub-37", "sub-39", "sub-43"]

condition_groups = {
    "Non-interleaved":     NN_INTERLEAVED,
    "First participants":  FIRST_PARTICIPANTs,
    "Second participants": SECOND_PARTICIPANTs,
}
condition_order = list(condition_groups.keys())
roi_names = ["front", "mid", "back"]

ATLAS = "HCPex_"
masks_output = r"/home/malberti/Unix_Folders/SWEEP2/Script/Miscellaneous"
os.makedirs(masks_output, exist_ok=True)

nib.save(cube_mask_front_img, (os.path.join(masks_output, f"HCPex_MNI_cube_mask-front.nii.gz")))
nib.save(cube_mask_mid_img, (os.path.join(masks_output, f"HCPex_MNI_cube_mask-mid.nii.gz")))
nib.save(cube_mask_back_img, (os.path.join(masks_output, f"HCPex_MNI_cube_mask-back.nii.gz")))

# ── Collect drift values ─────────────────────────────────────────────────────
records = []

for vp in vps:
    id = f"sub-{vp:02d}"

    condition = None
    for label, ids in condition_groups.items():
        if id in ids:
            condition = label
            break  

    for AP in APs:
        roi_means = {ses: {} for ses in sessions}
        ok = True

        for session in sessions:
            # NOTE: assumed metric-map naming, mirroring the per-AP mask naming.
            # Adjust this path if your actual layout differs. /home/malberti/Unix_Folders/SWEEP2/derivatives/sub-04/ses-01/dwi/index2MNI/Smoothing
            m_path = os.path.join(derivatives, id, session, "dwi", "index2MNI", "Smoothing", f"{id}_{session}_eddy-current_signal-drift_{AP}_corr_{metric}2MNI-HCPex6mm.nii")

            if not os.path.exists(m_path):
                print(f"✗  Missing metric map: {m_path}")
                ok = False
                break

            metric_data = nib.load(m_path).get_fdata()

            for roi in roi_names:
                mk_path = os.path.join(masks_output, f"{ATLAS}MNI_cube_mask-{roi}.nii.gz")
                if not os.path.exists(mk_path):
                    print(f"✗  Missing mask: {mk_path}")
                    ok = False
                    continue
                mask = nib.load(mk_path).get_fdata().astype(bool)
                roi_means[session][roi] = np.mean(metric_data[mask])

        if not ok:
            continue

        for roi in roi_names:
            if roi in roi_means["ses-01"] and roi in roi_means["ses-02"]:
                drift = roi_means["ses-02"][roi] - roi_means["ses-01"][roi]
                records.append({"subject": id, "AP": AP, "condition": condition,
                                 "ROI": roi, "drift": drift})

df = pd.DataFrame(records)
print(f"\nCollected {len(df)} rows | subjects: {df['subject'].nunique()}")

# ── Plot: one box plot per AP ─────────────────────────────────────────────────
for AP in APs:
    sub_df = df[df["AP"] == AP]
    if sub_df.empty:
        print(f"No data for AP {AP}, skipping plot")
        continue

    fig, ax = plt.subplots(figsize=(9, 6), dpi=300)
    sns.boxplot(data=sub_df, x="condition", y="drift", hue="ROI",
                order=condition_order, hue_order=roi_names, ax=ax)
    sns.stripplot(data=sub_df, x="condition", y="drift", hue="ROI",
                order=condition_order, hue_order=roi_names,
                dodge=True, palette={roi: "black" for roi in roi_names},
                alpha=0.5, size=4,
                ax=ax, legend=False)

    ax.axhline(0, color="grey", linestyle="--", linewidth=1)
    ax.set_xlabel("")
    ax.set_ylim(0.00003, -0.00003)
    ax.set_ylabel(f"{metric} drift (ses-02 − ses-01)")
    ax.set_title(f"{metric} drift by condition and ROI  |  AP = {AP}")
    ax.legend(title="ROI", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


    # ── Plot: one box plot per AP ─────────────────────────────────────────────────
roi_plot_order = ["mid", "front", "back"]   # order requested: mid, ant/front, back

for AP in APs:
    sub_df = df[df["AP"] == AP]
    if sub_df.empty:
        print(f"No data for AP {AP}, skipping plot")
        continue

    fig, ax = plt.subplots(figsize=(9, 6), dpi=300)
    sns.boxplot(data=sub_df, x="ROI", y="drift", hue="condition",
                order=roi_plot_order, hue_order=condition_order, ax=ax)
    sns.stripplot(data=sub_df, x="ROI", y="drift", hue="condition",
                  order=roi_plot_order, hue_order=condition_order,
                  dodge=True, palette={c: "black" for c in condition_order},
                  alpha=0.5, size=4,
                  ax=ax, legend=False)

    ax.axhline(0, color="grey", linestyle="--", linewidth=1)
    ax.set_xlabel("")
    ax.set_ylabel(f"{metric} drift (ses-02 − ses-01)")
    ax.set_ylim(0.00003, -0.00003)
    ax.set_title(f"{metric} drift by ROI and condition  |  AP = {AP}")
    ax.legend(title="Condition", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

# Plot B0-B1 and flip angle maps 
---------------------------------------------